In [5]:
import pickle
with open('./output/store/filtered_tokens.pkl', 'rb') as f:
    filtered_tokens, year = pickle.load(f)

In [6]:
seed = 42
year

'All'

In [ ]:
from gensim.models import LdaMulticore, Phrases
from gensim.corpora import Dictionary

no_below = 5
no_above = 0.25

bigram_model = Phrases(filtered_tokens, min_count=no_below, threshold=8)
trigram_model = Phrases(bigram_model[filtered_tokens], min_count=no_below, threshold=8)

trigram_tokens = [trigram_model[bigram_model[doc]] for doc in filtered_tokens]

dictionary = Dictionary(trigram_tokens)
dictionary.filter_extremes(no_below=no_below, no_above=no_above)
corpus = [dictionary.doc2bow(tokens) for tokens in trigram_tokens]
num_unique_tokens = len(dictionary)
print(f"Number of unique tokens: {num_unique_tokens}")

In [ ]:
with open('./output/store/corpus.pkl', 'wb') as f:
    pickle.dump((corpus, dictionary), f)

In [ ]:
from gensim.models import CoherenceModel
from sklearn.model_selection import train_test_split

topic_range = range(1, 30, 1)
evaluation_results = []

for num_topics in topic_range:
    print(f"Evaluating LDA model with {num_topics} topics...")

    train_corpus, test_corpus, train_texts, test_texts = train_test_split(
        corpus, trigram_tokens, test_size=0.4, random_state=seed, shuffle=True
    )

    lda_model = LdaMulticore(
        corpus=train_corpus,
        id2word=dictionary,
        num_topics=num_topics,
        iterations=200,
        random_state=seed,
        workers=4,  # Number of CPU cores to use
        passes=20,  # Number of passes through the corpus
        eval_every=None
    )

    perplexity = lda_model.log_perplexity(test_corpus, len(corpus))

    coherence_model = CoherenceModel(
        model=lda_model,
        texts=trigram_tokens,
        corpus=corpus,
        coherence='c_v'
    )
    coherence = coherence_model.get_coherence()

    model_metrics = {
        'n_topics': num_topics,
        'perplexity': perplexity,
        'coherence': coherence
    }

    evaluation_results.append(model_metrics)

    print(f"Number of Topics: {num_topics}, Perplexity: {perplexity:.4f}, Coherence: {coherence:.4f}")

print("\n=== Evaluation Results ===")
for result in evaluation_results:
    print(f"Number of Topics: {result['n_topics']}, Perplexity: {result['perplexity']:.4f}, Coherence: {result['coherence']:.4f}")

In [ ]:
import matplotlib.pyplot as plt

# Extract data from your evaluation results
num_topics = [result['n_topics'] for result in evaluation_results]
perplexity = [result['perplexity'] for result in evaluation_results]
coherence = [result['coherence'] for result in evaluation_results]

# Plot for Perplexity
plt.figure(figsize=(10, 7))
plt.plot(num_topics, perplexity, marker='o', color='tab:red', label='Perplexity')
plt.xlabel('Number of Topics')
plt.ylabel('Perplexity')
plt.title('LDA Model Evaluation: Perplexity vs. Number of Topics')
plt.legend()
plt.tight_layout()
plt.show()

# Plot for Coherence
plt.figure(figsize=(10, 7))
plt.plot(num_topics, coherence, marker='x', color='tab:blue', label='Coherence')
plt.xlabel('Number of Topics')
plt.ylabel('Coherence')
plt.title('LDA Model Evaluation: Coherence vs. Number of Topics')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Extract data from your evaluation results
num_topics = [result['n_topics'] for result in evaluation_results]
perplexity = [result['perplexity'] for result in evaluation_results]
coherence = [result['coherence'] for result in evaluation_results]

max_perp = max(perplexity)  # least negative value (worst perplexity)
min_perp = min(perplexity)  # most negative value (best perplexity)
norm_perplexity = [(max_perp - p) / (max_perp - min_perp) for p in perplexity]

plt.figure(figsize=(10, 7))
plt.plot(num_topics, norm_perplexity, marker='o', color='tab:red', label='Normalized Perplexity')
plt.plot(num_topics, coherence, marker='x', color='tab:blue', label='Coherence')
plt.xlabel('Number of Topics')
plt.ylabel('Score')
plt.title('LDA Model Evaluation')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import os
import pandas as pd
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

n_topics = [12]
lambda_val=0.6
top_n=30
    
for best_n_topics in n_topics:
    output = f"n_topics_{best_n_topics}_min"
    os.makedirs(f"./output/{output}", exist_ok=True)
    lda_model = LdaMulticore(
        corpus=corpus,
        id2word=dictionary,
        num_topics=best_n_topics,
        iterations=200,
        random_state=seed,
        workers=6,  # Number of CPU cores to use
        passes=20,  # Number of passes through the corpus
        eval_every=None,
    )
    
    output_html = f"./output/{output}/lda_model_{year}.html"
    vis_data = gensimvis.prepare(lda_model, corpus, dictionary, mds="mmds", sort_topics=True)
    
    topic_term_data = vis_data.topic_info
    topic_term_data = topic_term_data[topic_term_data['Category'] != 'Default'].copy()
    topic_term_data.loc[:,'Relevance'] = lambda_val * topic_term_data['logprob'] + (1 - lambda_val) * topic_term_data['loglift']
    topic_words = {}
    for topic in topic_term_data['Category'].unique():
        words = (
            topic_term_data[topic_term_data['Category'] == topic]
            .nlargest(top_n, 'Relevance')['Term']
            .tolist()
        )
        topic_words[topic] = words
    df = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in topic_words.items()]))
    df.to_excel(f"./output/{output}/LDA_TopWords.xlsx", index=False)
    pyLDAvis.save_html(vis_data, output_html)
    
    temp_file = f"/scratch/go76fil/Programs/Python/Paper_Topic_Modelling/output/{output}/model"
    lda_model.save(temp_file)